# Electrostatic terminals by connected net

`ElectrostaticSim` terminals select **connected conductors**, so two electrodes on
one layer are two terminals. This generic parallel-plate case checks that against
hand calculations:

- `m1` (z 0 to 0.5 um) and `m2` (z 1 to 1.5 um) are aluminum in SiO2 (relative
  permittivity 3.9) from z = -2 to 4 um.
- `LEFT` (0, 0)-(20, 20) and `RIGHT` (30, 0)-(50, 20) are separate electrodes on `m1`.
- `TOP` (0, 0)-(50, 20) on `m2` covers both, 0.5 um above them.
- A loose `m1` strip (0, 30)-(50, 35) is in no terminal, so it is grounded.

**What to expect.** The parallel-plate part of `C(TOP, LEFT)` is
`eps0 * 3.9 * 400 um^2 / 0.5 um` = 27.6 fF, and fringing adds to it. `C(TOP, RIGHT)`
is the same by symmetry, and `C(LEFT, RIGHT)` is small. Eliminating `TOP` as a
floating plate must give the series value
`C_TL * C_TR / (C_TL + C_TR + C_Tg)`.

Run the cells in order. The run needs Palace: use `sim.run_local()` with a local
executable or `sim.run()` on the GDSFactory+ cloud. Meshing and the config can be
checked without Palace.

In [ ]:
import gdsfactory as gf

from gsim.common.stack.extractor import Layer, LayerStack
from gsim.palace import ElectrostaticSim

LAYERS = {"m1": (1, 0.0, 0.5), "m2": (2, 1.0, 1.5)}  # GDS layer, zmin, zmax (um)

stack = LayerStack(
    layers={
        name: Layer(
            name=name,
            gds_layer=(number, 0),
            zmin=zmin,
            zmax=zmax,
            thickness=zmax - zmin,
            material="aluminum",
            layer_type="conductor",
        )
        for name, (number, zmin, zmax) in LAYERS.items()
    },
    materials={
        "aluminum": {"type": "conductor", "conductivity": 3.77e7},
        "sio2": {"type": "dielectric", "permittivity": 3.9},
        "air": {"type": "dielectric", "permittivity": 1.0},
    },
    dielectrics=[{"name": "ox", "zmin": -2.0, "zmax": 4.0, "material": "sio2"}],
)

gf.gpdk.PDK.activate()
c = gf.Component()
for layer, (x0, y0, x1, y1) in [
    ("m1", (0, 0, 20, 20)),  # LEFT
    ("m1", (30, 0, 50, 20)),  # RIGHT
    ("m2", (0, 0, 50, 20)),  # TOP
    ("m1", (0, 30, 50, 35)),  # strip: in no terminal, grounded
]:
    c.add_polygon([(x0, y0), (x1, y0), (x1, y1), (x0, y1)], layer=(LAYERS[layer][0], 0))
# a port names the net it sits on
c.add_port(
    name="TOP",
    center=(25, 10),
    width=1.0,
    orientation=0,
    layer=(2, 0),
    port_type="electrical",
)
c

### Configure the terminals

`TOP` is selected by the name of its port, `LEFT` and `RIGHT` by a region on `m1`.
`resolve_terminals()` shows which nets each terminal drives before anything is meshed.

In [ ]:
sim = ElectrostaticSim()
sim.set_output_dir("./palace-sim-electrostatic-nets")
sim.set_geometry(c)
sim.set_stack(stack)
sim.set_airbox(margin_x=20, margin_y=20, z_above=20, z_below=0)

sim.add_terminal("TOP", net="TOP")
sim.add_terminal("LEFT", region=(0, 0, 20, 20), layer="m1")
sim.add_terminal("RIGHT", region=(30, 0, 50, 20), layer="m1")

print(sim.nets())
for terminal in sim.resolve_terminals():
    print(terminal)
print(sim.validate_config())

### Mesh and config (job A)

Job A uses `refined_mesh_size=1.0`, `max_mesh_size=10.0` um. For the convergence run
(job B) change them to 0.5 and 5.0 and compare: the two should agree within 2 %.

In [ ]:
sim.mesh(refined_mesh_size=1.0, max_mesh_size=10.0)
sim.write_config()
print({e["name"]: len(e["phys_groups"]) for e in sim.mesh_groups["nets"]})
print(open("./palace-sim-electrostatic-nets/terminals.json").read())

### Run

Three excitations (`TOP`, `LEFT`, `RIGHT`). Use the cloud variant or a local Palace.

In [ ]:
results = sim.run()
# results = sim.run_local()

### Load the capacitances

In [ ]:
cap = sim.load_capacitance(results)
print((cap.maxwell_frame() * 1e15).round(3))
pairs = cap.pairs()
pairs["capacitance_fF"] = pairs["capacitance_F"] * 1e15
print(pairs[["first", "second", "capacitance_fF"]].round(3))
print("consistency checks:", cap.problems() or "ok")

EPS0 = 8.854187817e-12
parallel_plate_fF = EPS0 * 3.9 * 400e-12 / 0.5e-6 * 1e15
c_tl = cap.between("TOP", "LEFT") * 1e15
c_tr = cap.between("TOP", "RIGHT") * 1e15
c_lr = cap.between("LEFT", "RIGHT") * 1e15
print(f"parallel plate estimate: {parallel_plate_fF:.2f} fF (fringing adds to it)")
print(f"C(TOP, LEFT)  = {c_tl:.2f} fF, C(TOP, RIGHT) = {c_tr:.2f} fF")
print(f"|TL - TR| / TL = {abs(c_tl - c_tr) / c_tl:.2%}   (expect < 1 %)")
print(f"C(LEFT, RIGHT) / C(TOP, LEFT) = {c_lr / c_tl:.2%}   (expect < 2 %)")
print(f"C(LEFT, ground) = {cap.to_ground('LEFT') * 1e15:.3f} fF   (expect > 0)")

### Floating check

Treat `TOP` as a floating plate: the Schur complement removes it from the matrix.
The coupling left between `LEFT` and `RIGHT` must equal the series formula computed
by hand from the same matrix.

In [ ]:
floating_top = cap.eliminate(["TOP"])
c_tg = cap.to_ground("TOP") * 1e15
series_fF = c_tl * c_tr / (c_tl + c_tr + c_tg)
print(
    f"C(LEFT, RIGHT) with TOP floating = {floating_top.between('LEFT', 'RIGHT') * 1e15:.3f} fF"
)
print(f"by hand, C_TL*C_TR/(C_TL+C_TR+C_Tg) = {series_fF:.3f} fF")

### Optional: the IHP `cmim` cell

The `palace_electrostatic` notebook builds the 7 x 7 um or 10 x 10 um `cmim` capacitor with
`net="MINUS"` and `net="PLUS"`. Use `merge_via_distance=0` and an IHP release with the
split `mim_diel` / `mim` stack (gdsfactory/IHP#225) so the plate gap is right.